In [ ]:
import pandas as pd # Handling Data
import numpy as np # Handling Numbers
import seaborn as sns # Data Visuals
import matplotlib.pyplot as plt # Plots
from matplotlib.lines import lineStyles
from sklearn.preprocessing import StandardScaler
from scipy import stats

In [ ]:
print("==== Step 1: 'Import and Load' =====\n")
solo_df = pd.read_csv("C:\\Users\\Owner\\Desktop\\Comp Sci Year 3\\CS3072 Final Year Project\\Python\\solo_pubgdf.csv")
solo_df.head()
solo_df.info()

In [ ]:
print("==== Step 2: 'Check for Duplicates' =====\n")
solo_df.duplicated()

In [ ]:
print("==== Step 3: 'Identify Column Data Types' =====\n")
id_col = [col for col in solo_df.columns if solo_df[col].dtype == 'object']
num_col = [col for col in solo_df.columns if solo_df[col].dtype != 'object']

print('Id Columns:', id_col)
print('Numerical Columns:', num_col)

In [ ]:
print("==== Step 4: 'Count Unique Values in id_col' =====\n")
solo_df[id_col].nunique()

In [ ]:
# null_groupId = solo_df['groupId'].isnull().sum()
na_groupId = solo_df[solo_df['groupId'].isna()]
solo_df.groupby('groupId').size().sort_values(ascending=False).head()
# print("Null Counts:", null_groupId)
# print("Na Counts:", na_groupId)

In [ ]:
multi_group_ids = (
    solo_df
    .groupby('groupId')
    .size()
    .loc[lambda x: x > 1]
    .index
)

solo_group_anomalies = solo_df[solo_df['groupId'].isin(multi_group_ids)]

solo_group_anomalies.sort_values(['groupId', 'matchId'])

In [ ]:
solo_df_clean = solo_df[
    solo_df.groupby('groupId')['groupId'].transform('size') == 1
]
solo_df[id_col].nunique()

In [ ]:
print("==== Step 5: 'Calculate Missing Values as %' =====\n")
round((solo_df.isnull().sum() / solo_df.shape[0]) * 100, 2)

In [ ]:
print("==== Step 6: 'Drop Irrelevant or Data-Heavy Missing Columns' =====\n")
# solo_df.head()
# print(id_col)
solo_df.drop(columns=id_col) # Don't need the Id's for later steps

# No Data-heaving missing values present in this db

In [ ]:
print("==== Step 7: 'Detecting Outliers with Box Plots' =====\n")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Box plot of winPoints vs damageDealt', fontsize=14, fontweight='bold')

combat_cols = ['kills', 'assists', 'DBNOs', 'headshotKills', 'boosts']
active_players = solo_df[solo_df['damageDealt'] > 0]
non_zero_cols = solo_df[solo_df['kills'] > 0]

solo_df['log_kills'] = np.log1p(solo_df['kills'])
solo_df['log_assists'] = np.log1p(solo_df['assists'])
solo_df['log_DBNOs'] = np.log1p(solo_df['DBNOs'])
solo_df['log_hsKills'] = np.log1p(solo_df['headshotKills'])
solo_df['log_boosts'] = np.log1p(solo_df['boosts'])

log_cols = ['log_kills', 'log_assists', 'log_DBNOs', 'log_hsKills', 'log_boosts']

sns.boxplot(data=solo_df[log_cols], ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title('Distribution of Combat Stats')
axes[0, 0].set_ylabel('Count')

# 2. Box plot of Damage Dealt (Large float values)
# Plotted separately because its scale would squash the other plots
sns.boxplot(y=active_players['damageDealt'], ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Distribution of Damage Dealt from active_players')
axes[0, 1].set_ylabel('Damage')

# 2. Box plot of Damage Dealt (Large float values)
# Plotted separately because its scale would squash the other plots
sns.boxplot(x=non_zero_cols['kills'], ax=axes[1, 0], color='salmon')
axes[1, 0].set_title('Distribution of Damage Dealt from solo_df')
axes[1, 0].set_ylabel('Damage')

# # 3. Categorical Box plot: Win Place % vs Match Type
# # Useful to compare different groups defined by a categorical column
# sns.boxplot(x='matchType', y='winPlacePerc', data=active_players, ax=axes[1, 0], palette="viridis")
# axes[1, 0].set_title('Win Place Percentage by Match Type')
# axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Categorical Box plot: Kills vs Match Type
sns.boxplot(x='matchType', y='kills', data=active_players, ax=axes[1, 1], palette="coolwarm")
axes[1, 1].set_title('Kills Distribution by Match Type')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()



In [ ]:
# Handling Outliers
# print("Length of solo_df: ", len(solo_df))
# df_processed = solo_df.copy()
#
# min_valid = df_processed["killPoints"]%df_processed["kills"] > 1
# max_valid = df_processed["killPoints"]%df_processed["kills"] < 10
#
# print(min_valid)
# print(max_valid)
#
# invalid_rows = df_processed[
#     (((df_processed["killPoints"] == 0) & (df_processed["kills"] > 0)) |
#     ((df_processed["killPoints"] > 0) & (df_processed["kills"] == 0))) |
#     (min_valid & max_valid)
# ]
#
# df_processed.drop(invalid_rows.index, inplace=True)
# print("Length of df_processed: ", len(df_processed))
# print("Length of invalid_rows: ", len(invalid_rows))

In [ ]:
df_processed = solo_df.copy()

# 1. Calculate ratio safely
# We replace 0 with NaN temporarily to avoid division by zero errors/warnings,
# though your original method works if you ignore the Infinity warnings.
df_processed['kp_ratio'] = df_processed['killPoints'] / df_processed['kills'].replace(0, float('nan'))
print(df_processed['kp_ratio'])

# DEBUG STEP: Look at the actual ratios before filtering!
print("--- Ratio Statistics ---")
# This will show you the min, max, and average points per kill in your data
print(df_processed['kp_ratio'].describe())
print("-" * 20)

# 2. Adjusted Logic
# Example: If 'describe' shows the max ratio is 20, we should set the limit to 25 or 30
upper_limit = 30 # Adjust this based on the print output above

invalid_rows = df_processed[
    (df_processed["killPoints"] == 0) & ((df_processed["kills"] > 0) | (df_processed["assists"] > 0)) |
    (df_processed["kills"] == 0) & (df_processed["killPoints"] > 0)
    # ((df_processed["kills"] > 0) & (df_processed["killPoints"] == 0)) |
    # # Adjusted ratio check
    # (
    #     (df_processed["kills"] > 0) &
    #     ((df_processed['ratio'] < 1) | (df_processed['ratio'] > upper_limit))
    # )
]

df_processed.drop(invalid_rows.index, inplace=True)
# df_processed.drop(df_processed['killPoints'] == 1000)
print("Rows dropped:", len(invalid_rows))
print("Remaining rows:", len(df_processed))
df_processed.info()
df_processed.head()
print("Number of Matches:", len(df_processed['matchId'].unique()))
print("Player count for each match:", df_processed['matchId'].value_counts())

In [ ]:
random_match_id = pd.Series(df_processed['matchId'].unique()).sample(n=1).iloc[0]
single_match_df = df_processed[df_processed['matchId'] == random_match_id]
print(single_match_df.head())
print(f"Selected Match ID: {random_match_id}")
print(f"Total players in this match: {len(single_match_df)}")

In [ ]:
# Scatter Plot
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Scatter of Processed Solo Data', fontsize=20, fontweight='bold')

scatter_pairs = [
    ('kp_ratio', 'winPoints'),
    ('kills', 'killPoints'),
    ('assists', 'killPoints'),
    ('killPoints', 'assists')
]

for idx, (var1, var2) in enumerate(scatter_pairs):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]

    ax.scatter(df_processed[var1], df_processed[var2], label='Solo Data')

    ax.set_xlabel(var1)
    ax.set_ylabel(var2)
    ax.set_title(f'{var1} vs {var2}')
    ax.legend()
    # ax.grid(True, aplha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
    # The Correlation Heatmap
rows_to_use = ['assists', 'boosts', 'kills', 'killPoints', 'damageDealt', 'DBNOs', 'headshotKills', 'winPoints', 'winPlacePerc']
plt.figure(figsize=(12, 10))
matrix = df_processed[rows_to_use].corr()
sns.heatmap(matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation Matrix of Solo Features')
plt.show()

In [ ]:
# Feature Engineering - Creating Derived Features



